# A context playbook, evolved

The EvoSkill sibling evolves prose an agent reads whole. This one evolves a
*playbook*: canonical JSON whose entries carry stable logical identities, so a
single entry can be amended or retired without losing its history -- and without
rewriting the other entries.

Same boundary as the sibling, same deterministic fixture agent, no credential.
The [example README](https://github.com/salzubi401/meta-evolve/blob/main/examples/research/context_evolution/README.md) is the canonical explanation.

The whole lifecycle is one call, so that is what comes first. Everything after
it inspects the result.

In [1]:
import sys
from pathlib import Path

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
example = root / "examples/research/context_evolution"
if str(example) not in sys.path:
    sys.path.insert(0, str(example))

from ace_context import PLAYBOOK_FIXTURE_AGENT
from ace_study import study_playbook

study = study_playbook(PLAYBOOK_FIXTURE_AGENT)

print("selected:", study.selected.identities)
print("after consolidation:", study.playbook.identities)
print("held out:", study.held_out.solved)
print("failures:", study.failures)

selected: ('interest-element', 'scale-in-thousands', 'lease-element')
after consolidation: ('interest-element', 'lease-element')
held out: 1.0
failures: ()


## Why identity is the whole point

A playbook is one `meta.Text` holding canonical JSON -- sorted keys, no spare
whitespace -- so two equal playbooks have byte-identical digests. Each entry
keeps its identity across an amendment, and retiring one keeps it in the record
instead of deleting it.

In [2]:
import meta_evolve as meta

import playbook
import playbook_codec as codec
from ace_context import SEED_PLAYBOOK

encoded = codec.encode(SEED_PLAYBOOK)
print("schema:", playbook.SCHEMA)
print("seed items:", SEED_PLAYBOOK.identities)
print("round trip is byte-identical:",
      codec.encode(codec.decode(encoded)) == encoded)

item = SEED_PLAYBOOK.by_identity()["interest-element"]
amended = item.amended("Cash interest is tagged nwi:InterestPaidNetOfCapitalized.")
print("identity survives an amendment:", amended.identity == item.identity)
print("retiring keeps the record:", amended.retired().status)

schema: context-playbook-v1
seed items: ('interest-element', 'scale-in-thousands')
round trip is byte-identical: True
identity survives an amendment: True
retiring keeps the record: retired


## Three splits, two tagging policies

The same two policies appear in every split, over *differently worded*
disclosures. That is deliberate: an entry earns its keep by carrying one
filer's policy to a note nobody has read yet, so the private and held-out
rollouts must not be the development ones renamed.

Development failures are the only thing a proposer is allowed to learn from, and
only the ones that **finished and were wrong** -- a rollout that faulted is not
evidence, so it buys no feedback. The held-out split runs once, after
selection.

In [3]:
import ace_rollouts as rollouts

for split in (rollouts.DEVELOPMENT, rollouts.PRIVATE, rollouts.HELD_OUT):
    tasks = rollouts.rollouts_in(split)
    print(f"{split:12}", [(task.name, task.target, task.answer.strip())
                          for task in tasks])

development  [('debt-note', 'note_09_debt.tag', 'nwi:InterestPaidNetOfCapitalized'), ('lease-note', 'note_14_leases.tag', 'nwi:OperatingLeaseCostExcludingShortTerm')]
private      [('cash-flows', 'note_22_cash_flows.tag', 'nwi:InterestPaidNetOfCapitalized'), ('commitments', 'note_18_commitments.tag', 'nwi:OperatingLeaseCostExcludingShortTerm')]
held-out     [('borrowings', 'note_07_borrowings.tag', 'nwi:InterestPaidNetOfCapitalized'), ('right-of-use', 'note_11_right_of_use.tag', 'nwi:OperatingLeaseCostExcludingShortTerm')]


## What that one call did

Development, search, consolidation, held out -- in the one order they are valid
in. `study_playbook` owns that order, so a caller cannot skip the consolidation
re-score or run held-out before selection, and nothing below re-drives a phase:
it all reads off the `study` from the first cell.

The seed is a tagging manual that is already wrong in two different ways: it
names the standard element where this filer uses its own extension, and it
carries a presentation rule nothing in the evidence supports. So the first proposal has to
*curate* -- amend the rescuable entry, retire the unsupported one -- before the
second can grow the playbook.

The result also carries the *curator's declaration*. A local component is named
by module and qualname, so two closures from one factory compare equal; the
curator therefore declares a digest over the published lesson manifest, and two
runs given different lessons cannot record the same proposer and the same run
id while producing different playbooks.

In [4]:
from ace_context import PLAYBOOK_CALL_CEILING

print("development:", study.development.solved,
      "-> lessons:", [lesson.identity for lesson in study.lessons])
print("trial scores:", [trial.metrics["solved"] for trial in study.run.trials()])
print("active:", [item.identity for item in study.selected.active()])
print("consolidated:", study.consolidation.accepted, study.consolidation.reason,
      "->", study.consolidation.playbook.identities)

print("curator:", study.curator.name, "| named only:", study.curator.name_only)
print("nothing left name-only:", study.run.summary().name_only_components)
print("all four phases cost:", study.usage.evaluations, "evaluations and",
      study.usage.tokens, "reported tokens (the fixture reports none)")
print("worst-case wrapper calls:", PLAYBOOK_CALL_CEILING)

development: 0.0 -> lessons: ['interest-element', 'lease-element']
trial scores: [0.0, 0.5, 1.0]
active: ['interest-element', 'lease-element']
consolidated: True neutral -> ('interest-element', 'lease-element')
curator: context-evolution:playbook-curator | named only: False
nothing left name-only: ()
all four phases cost: 3 evaluations and 0 reported tokens (the fixture reports none)
worst-case wrapper calls: 14


## Four ways to be unrankable

A low score means *the playbook was wrong*. None of these do, so none of them is
allowed to be a low score. Two are about the candidate:

In [5]:
import ace_sessions as sessions
from ace_episodes import evaluate_playbook
from inner_agent import InnerAgent

invalid = evaluate_playbook(meta.Text("not a playbook at all"),
                            PLAYBOOK_FIXTURE_AGENT)
print("invalid ->", invalid.failure.details["kind"],
      "| metrics:", dict(invalid.metrics))


def silent_agent(_instructions):
    def propose(_tree, _context):
        raise RuntimeError("wrapper produced nothing")

    return propose


silent = sessions.run_split(rollouts.PRIVATE, study.selected,
                            InnerAgent(build=silent_agent, model="silent-v1",
                                       revision="silent-v1"))
print("wrapper ->", {name: fault.kind for name, fault in silent.unfinished.items()})

invalid -> invalid | metrics: {}
wrapper -> {'cash-flows': 'wrapper-failed', 'commitments': 'wrapper-failed'}


And two are about the *evidence* rather than the candidate. A file that is gone
cannot be judged, and an exhausted session is nobody's evidence about anything.

Both refusals happen where the rule is. The check itself refuses an absent
target rather than returning `0.0` and letting the caller relabel the total
afterwards -- which it used to, leaving the *inner* run holding a successful
evaluation of zero underneath an outcome that said unverifiable. `reported`
below is the library's own failure kind, so you can see the refusal crossing
the real boundary rather than being inferred from a missing file.

The timeout is a real `TimeoutFailure` for the same reason. An earlier version
of this example classified timeouts with a hand-written object whose `kind` no
shipped class carries, so a genuine timeout was reported as the wrapper's fault
and every test agreed.

In [6]:
from meta_evolve.domain import TimeoutFailure
from meta_evolve.ports import ProposalResult


def deleting_agent(_instructions):
    """Returns a candidate, but drops the file the check reads."""

    def propose(tree, _context):
        return type(tree)({"README.md": tree["README.md"]})

    return propose


def timing_out(_instructions):
    def propose(_tree, _context):
        return ProposalResult(failure=TimeoutFailure(
            message="the session exceeded its wall budget",
            details={"reason": "wall clock"},
        ))

    return propose


for name, build in (("deleted", deleting_agent), ("timed out", timing_out)):
    split = sessions.run_split(rollouts.PRIVATE, study.selected,
                               InnerAgent(build=build, model=f"{name}-v1",
                                          revision=f"{name}-v1"))
    print(f"{name} ->", {task: (fault.kind, fault.reported)
                         for task, fault in split.unfinished.items()})

deleted -> {'cash-flows': ('unverifiable', 'evaluator_failure'), 'commitments': ('unverifiable', 'evaluator_failure')}
timed out -> {'cash-flows': ('infrastructure-failed', 'timeout'), 'commitments': ('infrastructure-failed', 'timeout')}


## A split that did not finish has no score

One success beside one timeout is not `1.0`. Averaging whatever finished would
be a claim about the playbook that the evidence does not support, so a partial
split refuses to summarise itself -- and the study names the phase and the kind
instead.

In [7]:
import ace_outcomes as outcomes

calls = {"n": 0}


def flaky(instructions):
    """Fails one session, after selection is already over."""

    inner = PLAYBOOK_FIXTURE_AGENT.build(instructions)

    def propose(tree, context):
        calls["n"] += 1
        return inner(tree, context) if calls["n"] != 13 else ProposalResult(
            failure=TimeoutFailure(message="out of time",
                                   details={"reason": "wall clock"})
        )

    return propose


partial = study_playbook(
    InnerAgent(build=flaky, model="flaky-v1", revision="flaky-v1")
)
print("held-out complete:", partial.held_out.is_complete)
try:
    partial.held_out.solved
except outcomes.SplitIncomplete as refusal:
    print("refused:", refusal)
print("failures:", partial.failures)

held-out complete: False
refused: the held-out split did not finish: borrowings: infrastructure-failed (wall clock)
failures: (StudyFailure(phase='held-out', name='borrowings', kind='infrastructure-failed', detail='wall clock'),)


## Change and predict

This runs in a cell too. `revise` takes the published lessons and *nothing
else*, so you can hand it your own and watch curation decide -- no repository
file to edit, no kernel to restart.

Give it a lesson that contradicts an entry the seed already states, and predict
which of three things happens before you run it: does the mechanism amend that
entry, retire it, or append a second one that disagrees?

```python
from ace_curation import revise
from ace_rollouts import Lesson

contradiction = Lesson(
    identity="interest-element", section="debt", marker="nwi:InterestPaidGross",
    content="Cash interest is tagged nwi:InterestPaidGross, before capitalization.",
)
result = revise((contradiction,))(codec.encode(SEED_PLAYBOOK), None)
[(item.identity, item.status) for item in codec.decode(result.candidate).items]
```

Which of the three would you *want*? And if the proposal is refused instead,
read `result.failure` -- a refused transition is an outcome here, not a crash.

The EvoSkill sibling is in [`evoskill.ipynb`](https://github.com/salzubi401/meta-evolve/blob/main/examples/research/context_evolution/evoskill.ipynb).

Neither notebook reproduces its paper's system, benchmarks, transfer results, or
reported gains. They demonstrate a mechanism.